**Machine Learning & Neural Networks Midterm Project**

Comparing Decision Tree Algorithm to kNN (k-Nearest Neighbour) on the Cleveland Heart Disease Dataset

By Alex Shaw





I have chosen kNN and Decision Trees as the two algorithms I want to evaluate for reasons laid out in my report. I did a brief google search and came accross the Cleveland Heart Disease dataset which to my understanding has already been processed for Machine Learning which is ideal as there are no marks going to data cleaning in this project. 

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#tool used to split data into training and testing data
from sklearn.model_selection import train_test_split

#tool used for data scaling where necessary
from sklearn.preprocessing import StandardScaler

#tool to implement decision tree model
from sklearn.tree import DecisionTreeClassifier

#tool to test how accurate a given algorithm is
from sklearn.metrics import accuracy_score


In [4]:
#column headers taken from the heart-disease.names file documentation
column_names = [
    "age",          #patient age
    "sex",          #sex (1 = male, 0 = female)
    "cp",           #chest pain type (4 options)
    "trestbps",     #resting blood pressure
    "chol",         #cholesterol
    "fbs",          #fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
    "restecg",      #resting ecg results
    "thalach",      #maximum heart rate achieved
    "exang",        #exercise induced angina (1 = yes, 0 = no)
    "oldpeak",      #ST depression induced by exercise relative to rest
    "slope",        #slope of the peak exercise ST segment (3 options)
    "ca",           #number of major vessels (0 - 3) coloured by flouroscopy
    "thal",         #(3 = normal, 6 = fixed defect, 7 = reversible defect)
    "disease_present" #degree of heart disease (0 = none, 1–4 = increasing severity)
]

#load the dataset - there is no header row present so above are the column headers taken from documentation
df = pd.read_csv("processed.cleveland.data", names=column_names)

#display the first rows to confirm data has been loaded 
df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,disease_present
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [5]:
#NOTE: this cell was inserted once I reached the data scaling part of the project. 
#I realised there were data fields missing or reading as ?
#scaler from scikit can only handle numerical values so I had to remove rows with incomplete values.
#This was required to be completed BEFORE the data was split into 80/20 to ensure an even split
#Otherwise more data could be removed from one side than the other depending on the split

#replace missing values marked as '?' with NaN
df = df.replace("?", np.nan)

#remove rows that contain missing values
df = df.dropna()

#check the dataset shape after removing missing values
df.shape

(297, 14)

In [6]:
#separate the patient health measurements from the diagnosis result
#X contains information in the dataset headers
#these are the values used by the model to make a prediction
#axis = 1 removes column called "disease_present"
x = df.drop("disease_present", axis=1)

#Y contains the heart disease classification for each patient (0–4)
#this is what the model is trying to learn how to predict
y = df["disease_present"]

#check that the data has been split correctly into inputs and outputs
x.shape, y.shape


((297, 13), (297,))

The cell above shows output in line with what we wanted to see. X has 297 patients and 13 features whereas Y contains the corresponding diagnosis values for the 297 patients.

In [7]:
#shuffles and splits the dataset into training and testing sets

#the training set is used to train the algorithm to make predictions
#the testing set is used to evaluate how well the model has been trained

#x contains patient information, y contains the heart disease classification for the same patients

#test_size means that 20% of the data will be reserved for testing and the remaining 80% will be used for training

#random_state is an artbitrary integer to ensure the block of code is not randomised everytime it is run
#this ensures that the same patients who are part of the training data, stay there and vice versa
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

#display the shapes to confirm the split
x_train.shape, x_test.shape, y_train.shape, y_test.shape


((237, 13), (60, 13), (237,), (60,))

Above reads 237 patients used for training with 13 features.
60 patients are held back for testing with 13 features.
Diagnosis for the 237 and the 61 respective patients also executed correctly at this point.

Next we need to implement data scaling for kNN.

In [8]:
#scale the feature data so that all measurements are on a similar scale
#this is important for kNN because it relies on distance calculations

#initialise the scaler from scikit-learn
scaler = StandardScaler()

#fit the scaler using the training data and transform the training features into scaled versions of themselves
#this learns the scale from the training data only
x_train_scaled = scaler.fit_transform(x_train)

#transform the test data using the same scaling
#this ensures the test data is treated in the same way as the training data
x_test_scaled = scaler.transform(x_test)


In [9]:
#print some of the scaled data to ensure it is present and has been scaled
x_train_scaled[:5]


array([[-1.74167853, -1.52906121, -0.19776829,  0.31928358, -0.54421318,
        -0.43698372, -1.04980046,  0.15179817, -0.73413966, -0.91504143,
         0.62991241, -0.72057669, -0.91689986],
       [ 0.60111446, -1.52906121, -2.28092759,  0.98598146, -0.1617713 ,
        -0.43698372, -1.04980046,  0.97139911, -0.73413966, -0.12163356,
        -1.01062871, -0.72057669, -0.91689986],
       [ 1.60516859, -1.52906121, -2.28092759,  0.4303999 , -0.1808934 ,
        -0.43698372, -1.04980046,  0.10866128, -0.73413966,  0.67177432,
        -1.01062871,  1.38777733, -0.91689986],
       [ 0.37799131,  0.65399606, -1.23934794, -0.68076324,  0.67960083,
        -0.43698372,  0.96479637,  0.4968933 , -0.73413966,  0.67177432,
         0.62991241, -0.72057669, -0.91689986],
       [-0.84918597,  0.65399606, -0.19776829, -0.12518167,  0.08681592,
        -0.43698372, -1.04980046,  1.31649423, -0.73413966, -0.91504143,
        -1.01062871, -0.72057669, -0.91689986]])

Now to implement the Decision Tree algorithm using the scikit-learn python library.

In [10]:
#initialise a Decision Tree classifier from the scikit-learn library
#this model learns a set of decision rules (nodes) from the training data
#max_depth limits how complex the tree can become to reduce overfitting
#random_state is assigned an integer to ensure that the notebook is not completely random each time it is run
dt = DecisionTreeClassifier(max_depth=5, random_state=42)

#train the Decision Tree using the training data
#the model looks over patient symptoms alongside the diagnosis and learns to predict in the future.
#decision trees do not require scaled data, so the original values are used
dt.fit(x_train, y_train)

#use the trained Decision Tree to predict heart disease classification
#for patients in the test dataset that the model has not seen before
y_pred_dt = dt.predict(x_test)

#calculate how often the Decision Tree predictions match the true diagnoses
#accuracy is the proportion of correct predictions
dt_accuracy = accuracy_score(y_test, y_pred_dt)

#display the accuracy score for the Decision Tree model
dt_accuracy


0.5833333333333334

Now we will implement a "from scratch" implementation of kNN. Using euclidean distance, we will make a loop which will consider a test patient, diagnosis unknown, and then looking at the medical features and symptoms we take the k-Nearest Neighbours of that patient based on their features, and when their diagnosis is revealed, we predict the diagnosis of the test patient. The loop will continue as such predicting the diagnosis of all test patients.

In [11]:
#implement k-Nearest Neighbours from scratch using numpy
#this algorithm predicts heart disease severity by comparing patients to similar past cases
#the output classes range from 0 to 4, representing increasing severity
#the euclidean distance based on features is calculated and then the closest neighbours with positive diagnosis are used to predict

#function to calculate euclidean distance between two patients
#this measures how similar two patients are based on their feature values not considering diagnosis
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

#function to extract the distance value from a (distance, label) tuple
#this allows the distances list to be sorted correctly so that other features are not mixed up
#only the distance is considered
def get_distance(item):
    return item[0]

#function that performs kNN prediction
#x_train is made up of training features
#y_train is made up of training diagnosis
#x_test contains patients we want to predict
#k controls how many neighbours are used for each prediction
def knn_predict(x_train, y_train, x_test, k):
    
    #create an empty list to store predictions for all test patients
    predictions = []

    #loop through each test patient
    for test_point in x_test:
        
        #create an empty list to store distances to all training patients
        distances = []

        #loop through every training patient
        #each training patient already has a known diagnosis
        for i, train_point in enumerate(x_train):
            
            #calculate how far the test patient is from this training patient
            #distance based on symptoms is calculated
            distance = euclidean_distance(test_point, train_point)
            
            #store the distance together with the training patient’s diagnosis
            #this puts together the distances with the diagnosis so they are permanently connected
            distances.append((distance, y_train[i]))

        #sort all training patients by distance
        #closest patients (most similar) come first
        distances.sort(key=get_distance)

        #select only the first k patients after sorting
        #these are the k most similar patients to the test patient
        k_nearest = distances[:k]
        
        #extract just the diagnoses (0 - 4) from the k nearest patients
        #distance values are no longer needed for decision making as we have already selected the k slosest patients
        k_labels = [label for _, label in k_nearest]
        
        #count how often each diagnosis appears among the neighbours
        #choose the most common diagnosis (majority vote)
        prediction = max(set(k_labels), key=k_labels.count)
        
        #store the predicted diagnosis for this test patient
        predictions.append(prediction)

    #after all test patients have been processed
    #return the list of predicted diagnoses
    return predictions


#choose the number of neighbours to use
k = 5

#run the kNN algorithm using scaled training and test data
#this produces a prediction for every test patient
#only required for kNN and not for Decision Trees
y_pred_knn = knn_predict(x_train_scaled, y_train.values, x_test_scaled, k)

#calculate accuracy by comparing predictions to true diagnoses
#this computes the proportion of correct predictions
knn_accuracy = np.mean(y_pred_knn == y_test.values)

#output the final accuracy value
knn_accuracy


0.6

Both ML models are reasonably accurate with a 1/2 chance of accurately predicting the correct diagnosis. However, in the interest of tailoring these models to find the best conditions to run with this particular dataset, I am now going to attempted to put them into loops, one by one, and test the Decision Tree depth and k for kNN to see what the sweet spot might be in the Bias-Variance.

Firstly let's do Decision Tree depth loop. As it uses heavily the Scikit-Learn tool kit I imagine it will be more straight forward.

In [12]:
#test different tree depths to observe how model complexity affects accuracy
#depth values increase in steps of 1 to clearly show bias-variance behaviour
for depth in range(1, 20, 1):
    
    #initialise a Decision Tree with the current depth
    #changing max_depth controls how complex the tree is allowed to become
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    #train the Decision Tree on the training data
    #the model learns decision rules based on patient features
    dt.fit(x_train, y_train)
    
    #predict heart disease severity for unseen test patients
    y_pred_dt = dt.predict(x_test)
    
    #calculate accuracy for this specific tree depth
    #this measures how well the model generalises to new data
    dt_accuracy = accuracy_score(y_test, y_pred_dt)
    
    #display the depth value alongside its corresponding accuracy
    print(f"DT_depth = {depth}, accuracy = {dt_accuracy:.3f}")


DT_depth = 1, accuracy = 0.500
DT_depth = 2, accuracy = 0.600
DT_depth = 3, accuracy = 0.633
DT_depth = 4, accuracy = 0.583
DT_depth = 5, accuracy = 0.583
DT_depth = 6, accuracy = 0.550
DT_depth = 7, accuracy = 0.533
DT_depth = 8, accuracy = 0.517
DT_depth = 9, accuracy = 0.500
DT_depth = 10, accuracy = 0.500
DT_depth = 11, accuracy = 0.500
DT_depth = 12, accuracy = 0.483
DT_depth = 13, accuracy = 0.483
DT_depth = 14, accuracy = 0.500
DT_depth = 15, accuracy = 0.483
DT_depth = 16, accuracy = 0.483
DT_depth = 17, accuracy = 0.483
DT_depth = 18, accuracy = 0.483
DT_depth = 19, accuracy = 0.483


As shown above, the Decision Tree model with a depth of 1 underfits the data, achieving an accuracy of only 50%. Increasing the tree depth reduces bias and improves performance, with accuracy rising to 60% at depth 2 and peaking at 63.3% at depth 3, which represents the optimal “sweet spot”. Beyond this point, accuracy declines as the model becomes increasingly complex and begins to overfit the training data, before eventually stabilising at a lower level. Notably, the initial evaluation of the model at a maximum depth of 5 produced an accuracy of 58%, whereas systematic hyperparameter testing improved accuracy to 63.3%. This demonstrates the importance of model tuning. Further improvements may be possible through feature selection or engineering, such as adding relevant patient attributes or removing redundant features.

Now I will apply the same looping principle to the kNN model and see what the ideal value for k is to return the highest accuracy.

In [15]:
#test different values of k to observe how neighbourhood size affects accuracy
#k values increase gradually to analyse bias-variance behaviour
for k in range(1, 21):
    
    #run the kNN algorithm using the current value of k
    #scaled features are required for distance-based models
    y_pred_knn = knn_predict(x_train_scaled, y_train.values, x_test_scaled, k)
    
    #calculate accuracy for this specific value of k
    #this measures how well the model generalises to unseen data
    knn_accuracy = np.mean(y_pred_knn == y_test.values)
    
    #display the value of k alongside its corresponding accuracy
    print(f"k = {k}, accuracy = {knn_accuracy:.3f}")


k = 1, accuracy = 0.683
k = 2, accuracy = 0.700
k = 3, accuracy = 0.600
k = 4, accuracy = 0.650
k = 5, accuracy = 0.600
k = 6, accuracy = 0.633
k = 7, accuracy = 0.633
k = 8, accuracy = 0.633
k = 9, accuracy = 0.650
k = 10, accuracy = 0.633
k = 11, accuracy = 0.667
k = 12, accuracy = 0.633
k = 13, accuracy = 0.633
k = 14, accuracy = 0.633
k = 15, accuracy = 0.617
k = 16, accuracy = 0.600
k = 17, accuracy = 0.617
k = 18, accuracy = 0.617
k = 19, accuracy = 0.633
k = 20, accuracy = 0.650


The k-Nearest Neighbours results demonstrate a clear bias–variance trade-off. At k = 1 the model performs less than optimally which could indicate an overfitting of the data. At k = 2, the model exhibits low bias and high variance, as predictions are based on very local neighbourhoods. Accuracy peaks at 70% here, indicating that the dataset contains strong local structure and that limited smoothing is beneficial. As k increases, predictions are averaged over larger neighbourhoods, increasing bias and leading to underfitting, which results in reduced accuracy. The fact that accuracy peaks early on at k = 2 it would suggest that excessive smoothing masks important patterns in the data rather than improving classification.

**Results Comparison**

The results of the Decision Tree and k-Nearest Neighbours models highlight how different algorithms respond to model complexity and the bias–variance trade-off with regard to the underfitting of overly simple models and the overfitting of overly complex ones. The Decision Tree achieved its highest accuracy at a shallow depth of 3, after which performance declined as depth increased, indicating overfitting as the model became increasingly complex. In contrast, k-Nearest Neighbours performed best with a very small neighbourhood size, peaking at k = 2, with accuracy decreasing as k increased due to increased bias introduced by excessive smoothing. This contrast arises because, although both models perform best at small hyperparameter values, those values represent different levels and forms of complexity relative to how each algorithm learns.

Although the Decision Tree exhibited more stable performance across a range of depth values, kNN maintained comparatively high accuracy across the first 20 values of k and achieved superior peak performance. This suggests that, for this dataset and task, kNN is the more effective algorithm. Notably, both models performed best at lower values of their respective hyperparameters, reinforcing the idea that increased model complexity does not necessarily improve classification. Identifying the optimal “sweet spot” between bias and variance resulted in a significant improvement in accuracy for both models, demonstrating the importance of exploring the variation of model complexity and tweaking of models where possible.

While both models achieved their best performance at small hyperparameter values, these values represent different levels of complexity relative to each algorithm. For kNN, a value of k = 2 means that each prediction is based on only two neighbouring patients, making the model highly local and sensitive to small variations in the data, even though it performs well for this dataset. In contrast, a decision tree with a depth of 3 can already make several meaningful splits and capture interactions between features, providing sufficient structure to generalise without excessive complexity. Therefore, although both values are numerically small, they correspond to different degrees of model expressiveness, explaining why kNN performs best at k = 2 while the decision tree performs best at a depth of 3.

It is worth noting that the Decision Tree was outperformed by kNN because this dataset appears to work better when patients are compared directly to other similar patients, rather than being separated by a small number of global rules. The kNN model makes predictions by looking at the most similar patients and using their diagnoses, which allows it to capture small, detailed patterns in the data. In contrast, a shallow decision tree must rely on a limited number of splits, meaning some patients with different outcomes may still be grouped together. For example, two patients might have very similar symptoms but fall on opposite sides of a single split in a decision tree, whereas kNN would correctly treat them as similar cases. Because of this, kNN was able to make more accurate predictions for this dataset by using local patient similarities rather than broad decision rules.
